# 01 — Exploratory Data Analysis
**Goal:** Understand the shape, size, and content of the raw Medicaid dataset before any cleaning.

**Key Questions:**
- How many rows and columns are there?
- What are the data types?
- What is the date range?
- How many unique providers and HCPCS codes exist?
- What does the spending distribution look like?

In [1]:
import duckdb
import polars as pl
import os
import numpy as np
import pandas as pd

con = duckdb.connect()

parquet_path = os.path.expanduser("~/Downloads/medicaid-provider-spending.parquet")
csv_path = "/Users/adilkassim/Downloads/NPPES_Data_Dissemination_February_2026_V2/npidata_pfile_20050523-20260208.csv"

con.execute(f"CREATE VIEW medicaid AS SELECT * FROM read_parquet('{parquet_path}')")
con.execute(
    f"CREATE OR REPLACE VIEW provider_info_raw AS SELECT * FROM read_csv_auto('{csv_path}', all_varchar=1)"
)

In [ ]:
## Number of Rows in The Dataset
con.sql("SELECT COUNT(*) FROM MEDICAID")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│    227083361 │
└──────────────┘

Let's take a peek at how the data looks


In [4]:
con.sql("SELECT * FROM medicaid LIMIT 10")

┌──────────────────────────┬────────────────────────────┬────────────┬──────────────────┬────────────────────────────┬──────────────┬──────────────┐
│ BILLING_PROVIDER_NPI_NUM │ SERVICING_PROVIDER_NPI_NUM │ HCPCS_CODE │ CLAIM_FROM_MONTH │ TOTAL_UNIQUE_BENEFICIARIES │ TOTAL_CLAIMS │  TOTAL_PAID  │
│         varchar          │          varchar           │  varchar   │     varchar      │           int64            │    int64     │    double    │
├──────────────────────────┼────────────────────────────┼────────────┼──────────────────┼────────────────────────────┼──────────────┼──────────────┤
│ 1376609297               │ 1376609297                 │ T1019      │ 2024-07          │                      39765 │      1205701 │ 118887675.31 │
│ 1376609297               │ 1376609297                 │ T1019      │ 2024-08          │                      39677 │      1152534 │ 115561066.11 │
│ 1376609297               │ 1376609297                 │ T1019      │ 2024-05          │                 

## 1. Schema & Data Types

In [3]:
con.sql("DESCRIBE medicaid").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""BILLING_PROVIDER_NPI_NUM""","""VARCHAR""","""YES""",null,null,null
"""SERVICING_PROVIDER_NPI_NUM""","""VARCHAR""","""YES""",null,null,null
"""HCPCS_CODE""","""VARCHAR""","""YES""",null,null,null
"""CLAIM_FROM_MONTH""","""VARCHAR""","""YES""",null,null,null
"""TOTAL_UNIQUE_BENEFICIARIES""","""BIGINT""","""YES""",null,null,null
"""TOTAL_CLAIMS""","""BIGINT""","""YES""",null,null,null
"""TOTAL_PAID""","""DOUBLE""","""YES""",null,null,null


In [19]:
con.sql("SUMMARIZE medicaid").pl()

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""BILLING_PROVIDER_NPI_NUM""","""VARCHAR""","""0""","""SD07819990""",712549,null,null,null,null,null,227083361,0.00
"""SERVICING_PROVIDER_NPI_NUM""","""VARCHAR""","""0""","""X110067543""",1500685,null,null,null,null,null,227083361,4.18
"""HCPCS_CODE""","""VARCHAR""","""-""","""ZZZZZ""",12009,null,null,null,null,null,227083361,0.00
"""CLAIM_FROM_MONTH""","""VARCHAR""","""2018-01""","""2024-12""",93,null,null,null,null,null,227083361,0.00
"""TOTAL_UNIQUE_BENEFICIARIES""","""BIGINT""","""12""","""159423""",23052,"""49.84054224474861""","""316.1932694375741""","""15""","""22""","""39""",227083361,0.00
"""TOTAL_CLAIMS""","""BIGINT""","""12""","""1607071""",43184,"""82.90155619107645""","""1021.2468581306315""","""18""","""28""","""54""",227083361,0.00
"""TOTAL_PAID""","""DOUBLE""","""-183021.84""","""118887675.31""",8219815,"""4815.6889553501705""","""91195.11944330868""","""88.62291734503162""","""616.794328023858""","""1955.2135911204098""",227083361,0.00


In [5]:
con.sql("""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT HCPCS_CODE)      AS unique_hcpcs,
        COUNT(DISTINCT BILLING_PROVIDER_NPI_NUM) AS unique_providers,
        MIN(CLAIM_FROM_MONTH)           AS earliest_month,
        MAX(CLAIM_FROM_MONTH)           AS latest_month
    FROM medicaid
""").pl()

total_rows,unique_hcpcs,unique_providers,earliest_month,latest_month
i64,i64,i64,str,str
227083361,10881,617503,"""2018-01""","""2024-12"""


In [6]:
claims_and_paid = con.sql("""
    SELECT
        SUM(TOTAL_PAID) as total_paid,
        SUM(TOTAL_CLAIMS) as total_claims
        
    FROM medicaid
""").pl()

claims_and_paid

total_paid,total_claims
f64,"decimal[38,0]"
1.0936e12,18825564012


1.09 Trillion Dollars spent and 18.8 Billion Claims over 7 years 


In [7]:
total_days = 365 * 7


spend_per_day = claims_and_paid["total_paid"].item() / total_days
claims_per_day = claims_and_paid["total_claims"].item() / total_days


print(f"Roughly  ${round(spend_per_day, 2)} is spent per day")
print(f"There are roughly {round(claims_per_day, 2)} claims per day ")

Roughly  $428008936.8 is spent per day
There are roughly 7368126.81 claims per day 


In [21]:
con.sql("""
    SELECT Count(*)
    From medicaid
    Where Billing_Provider_NPI_NUM <> SERVICING_PROVIDER_NPI_NUM
""").pl()

count_star()
i64
147861529


In [22]:
con.sql("""
    SELECT Count(*)
    From medicaid
    Where Billing_Provider_NPI_NUM == SERVICING_PROVIDER_NPI_NUM
""").pl()

count_star()
i64
69731487


In [23]:
con.sql("""
    SELECT Count(*)
    From medicaid
    Where SERVICING_PROVIDER_NPI_NUM is null
""").pl()

count_star()
i64
9490345


Interestingly enough 

## 3. Spending Distribution

In [8]:
con.sql("""
    SELECT
        MIN(TOTAL_PAID)    AS min_paid,
        MAX(TOTAL_PAID)    AS max_paid,
        AVG(TOTAL_PAID)    AS avg_paid,
        MEDIAN(TOTAL_PAID) AS median_paid,
        SUM(TOTAL_PAID)    AS total_paid
    FROM medicaid
""").pl()

min_paid,max_paid,avg_paid,median_paid,total_paid
f64,f64,f64,f64,f64
-183021.84,1.1889e8,4815.688955,616.39,1.0936e12


In [9]:
## Let's check how many negatives there are
## We shouldn't remove negatives as they represent claim reversals
## and they ultimately represent net medicaid spend

con.sql("""
    SELECT 
        SUM(TOTAL_PAID),
        COUNT(TOTAL_PAID),
        SUM(TOTAL_PAID) / COUNT(TOTAL_PAID) as avg_reversal
    FROM medicaid
    WHERE TOTAL_PAID < 0
""")


┌────────────────────┬───────────────────┬────────────────────┐
│  sum(TOTAL_PAID)   │ count(TOTAL_PAID) │    avg_reversal    │
│       double       │       int64       │       double       │
├────────────────────┼───────────────────┼────────────────────┤
│ -8360301.790000003 │              9239 │ -904.8924981058559 │
└────────────────────┴───────────────────┴────────────────────┘

Only a small amount, roughly 8.3 million out of 

## 4. Top 20 HCPCS Codes by Volume

In [10]:
con.sql("SELECT COUNT(DISTINCT HCPCS_CODE) FROM medicaid")

┌────────────────────────────┐
│ count(DISTINCT HCPCS_CODE) │
│           int64            │
├────────────────────────────┤
│                      10881 │
└────────────────────────────┘

In [11]:
hcpcs = con.sql("""
    SELECT HCPCS_CODE, COUNT(*) AS count, SUM(TOTAL_PAID) AS total_paid
    FROM medicaid
    GROUP BY HCPCS_CODE
    ORDER BY count DESC
""").pl()
hcpcs

HCPCS_CODE,count,total_paid
str,i64,f64
"""99213""",13566914,3.3003e10
"""99214""",11728025,2.9914e10
"""99284""",3980198,2.0152e10
"""99283""",3270001,1.6875e10
"""99285""",2901462,1.5097e10
…,…,…
"""38241""",1,0.0
"""PE100""",1,0.0
"""4240F""",1,20.0


In [12]:
hcpcs["total_paid"].sum()

1093562833512.4133

In [13]:
hcpcs.head(32)["total_paid"].sum()

233645519908.75406

In [14]:
hcpcs_sorted_by_paid = hcpcs.sort("total_paid", descending=True)
hcpcs_sorted_by_paid.head(20)


HCPCS_CODE,count,total_paid
str,i64,f64
"""T1019""",457916,1.2274e11
"""T1015""",2429393,4.9153e10
"""T2016""",81288,3.4905e10
"""99213""",13566914,3.3003e10
"""S5125""",214380,3.1342e10
…,…,…
"""H2017""",292828,8.5404e9
"""T1017""",283832,8.4237e9
"""T1020""",39648,8.2126e9


In [15]:
hcpcs["count"].describe()


statistic,value
str,f64
"""count""",10881.0
"""null_count""",0.0
"""mean""",20869.714273
"""std""",209489.713986
"""min""",1.0
"""25%""",14.0
"""50%""",158.0
"""75%""",2199.0
"""max""",1.3566914e7


In [16]:
hcpcs_sorted = hcpcs.sort("total_paid", descending=True).with_columns(
    [
        (pl.col("total_paid").cum_sum() / pl.col("total_paid").sum()).alias(
            "cumulative_share"
        )
    ]
)

threshold_codes = {}
for threshold in [0.50, 0.80, 0.90, 0.95, 0.99]:
    n = (hcpcs_sorted["cumulative_share"] <= threshold).sum() + 1
    codes = hcpcs_sorted.head(n)["HCPCS_CODE"].to_list()
    threshold_codes[threshold] = codes
    print(
        f"{threshold:.0%} of claims covered by top {n} codes ({n / len(hcpcs_sorted):.1%} of all codes)"
    )

hcpcs_sorted

50% of claims covered by top 32 codes (0.3% of all codes)
80% of claims covered by top 177 codes (1.6% of all codes)
90% of claims covered by top 375 codes (3.4% of all codes)
95% of claims covered by top 687 codes (6.3% of all codes)
99% of claims covered by top 1780 codes (16.4% of all codes)


HCPCS_CODE,count,total_paid,cumulative_share
str,i64,f64,f64
"""T1019""",457916,1.2274e11,0.112238
"""T1015""",2429393,4.9153e10,0.157185
"""T2016""",81288,3.4905e10,0.189104
"""99213""",13566914,3.3003e10,0.219283
"""S5125""",214380,3.1342e10,0.247944
…,…,…,…
"""C9363""",1,0.0,1.0
"""A4293""",1,0.0,1.0
"""38241""",1,0.0,1.0


In [17]:
hcpcs_sorted = hcpcs.sort("count", descending=True).with_columns(
    [(pl.col("count").cum_sum() / pl.col("count").sum()).alias("cumulative_share")]
)

threshold_codes = {}
for threshold in [0.50, 0.80, 0.90, 0.95, 0.99]:
    n = (hcpcs_sorted["cumulative_share"] <= threshold).sum() + 1
    codes = hcpcs_sorted.head(n)["HCPCS_CODE"].to_list()
    threshold_codes[threshold] = codes
    print(
        f"{threshold:.0%} of claims covered by top {n} codes ({n / len(hcpcs_sorted):.1%} of all codes)"
    )

hcpcs_sorted.select(["HCPCS_CODE", "count", "cumulative_share"]).head(50)

50% of claims covered by top 74 codes (0.7% of all codes)
80% of claims covered by top 373 codes (3.4% of all codes)
90% of claims covered by top 744 codes (6.8% of all codes)
95% of claims covered by top 1232 codes (11.3% of all codes)
99% of claims covered by top 2675 codes (24.6% of all codes)


HCPCS_CODE,count,cumulative_share
str,i64,f64
"""99213""",13566914,0.059744
"""99214""",11728025,0.111391
"""99284""",3980198,0.128918
"""99283""",3270001,0.143318
"""99285""",2901462,0.156095
…,…,…
"""70450""",926017,0.408883
"""96372""",917535,0.412923
"""81025""",885615,0.416823


- very big spread, std deviation is 209,490
    - which makes sense give the difference between the min value (1) and the max calue (13 million)
- mean is significantly larger than the median, some codes are doing a lot of carrying  

## 5. Spending by Year (Quick Plot)

In [18]:
df_yearly["ratio"] = (
    df_yearly["total_paid_billions"] / df_yearly["total_claims_millions"]
)

NameError: name 'df_yearly' is not defined

In [ ]:
df_yearly

In [ ]:
import matplotlib.pyplot as plt

df_yearly = con.sql("""
    SELECT
        SUBSTR(CLAIM_FROM_MONTH, 1, 4) AS year,
        SUM(TOTAL_PAID) / 1e9          AS total_paid_billions,
        SUM(TOTAL_CLAIMS) / 1e6        AS total_claims_millions   -- scale down too
    FROM medicaid
    GROUP BY year
    ORDER BY year
""").df()

fig, ax1 = plt.subplots()

# Left Y-axis — Total Paid
color1 = "steelblue"
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Paid (Billions $)", color=color1)
ax1.plot(
    df_yearly["year"],
    df_yearly["total_paid_billions"],
    color=color1,
    marker="o",
    label="Total Paid",
)
ax1.tick_params(axis="y", labelcolor=color1)

# Right Y-axis — Total Claims
ax2 = ax1.twinx()
color2 = "orange"
ax2.set_ylabel("Total Claims (Millions)", color=color2)
ax2.plot(
    df_yearly["year"],
    df_yearly["total_claims_millions"],
    color=color2,
    marker="s",
    linestyle="--",
    label="Total Claims",
)
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("Medicaid Spending & Claims by Year")
fig.tight_layout()
plt.show()

years = df_yearly["year"]
x = np.arange(len(years))  # position of each group
width = 0.35  # width of each bar
fig, ax1 = plt.subplots(figsize=(10, 6))
# Left Y-axis — Total Paid (blue bars)
bars1 = ax1.bar(
    x - width / 2,
    df_yearly["total_paid_billions"],
    width,
    label="Total Paid (Billions $)",
    color="steelblue",
)
ax1.set_ylabel("Total Paid (Billions $)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
# Right Y-axis — Total Claims (red bars)
ax2 = ax1.twinx()
bars2 = ax2.bar(
    x + width / 2,
    df_yearly["total_claims_millions"],
    width,
    label="Total Claims (Millions)",
    color="tomato",
    alpha=0.8,
)
ax2.set_ylabel("Total Claims (Millions)", color="tomato")
ax2.tick_params(axis="y", labelcolor="tomato")
# Labels & formatting
ax1.set_xlabel("Year")
ax1.set_xticks(x)
ax1.set_xticklabels(years)
plt.title("Medicaid Spending & Claims by Year")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.95))
fig.tight_layout()
plt.show()

In [ ]:
df_yearly = con.sql("""
    SELECT
        SUBSTR(CLAIM_FROM_MONTH, 1, 4)       AS year,
        ABS(SUM(TOTAL_PAID))          AS total_recouped_millions
    FROM medicaid
    WHERE TOTAL_PAID < 0
    GROUP BY year
    ORDER BY year
""").df()

ax = df_yearly.plot(
    kind="line",
    x="year",
    y="total_recouped_millions",
    title="Total Medicaid Recoupments & Reversals by Year",
    legend=False,
)
ax.set_ylabel("Total Recouped (Millions $)")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()
